# Win or Lose - League of Legends (League of DSC)

**Name(s)**: Chaylen Nguyen-Tran, Jeffrey Kang

**Website Link**: https://github.com/chaylennn/Win-or-Lose-LoL

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

# from dsc80_utils import * # Feel free to uncomment and use this.

In [3]:
df = pd.read_csv("2025_LoL_esports_match_data_from_OraclesElixir.csv")

C:\Users\chayl\AppData\Local\Temp\ipykernel_26960\544348570.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("2025_LoL_esports_match_data_from_OraclesElixir.csv")


In [4]:
for i in df.columns:
    print(i)

gameid
datacompleteness
url
league
year
split
playoffs
date
game
patch
participantid
side
position
playername
playerid
teamname
teamid
firstPick
champion
ban1
ban2
ban3
ban4
ban5
pick1
pick2
pick3
pick4
pick5
gamelength
result
kills
deaths
assists
teamkills
teamdeaths
doublekills
triplekills
quadrakills
pentakills
firstblood
firstbloodkill
firstbloodassist
firstbloodvictim
team kpm
ckpm
firstdragon
dragons
opp_dragons
elementaldrakes
opp_elementaldrakes
infernals
mountains
clouds
oceans
chemtechs
hextechs
dragons (type unknown)
elders
opp_elders
firstherald
heralds
opp_heralds
void_grubs
opp_void_grubs
firstbaron
barons
opp_barons
atakhans
opp_atakhans
firsttower
towers
opp_towers
firstmidtower
firsttothreetowers
turretplates
opp_turretplates
inhibitors
opp_inhibitors
damagetochampions
dpm
damageshare
damagetakenperminute
damagemitigatedperminute
damagetotowers
wardsplaced
wpm
wardskilled
wcpm
controlwardsbought
visionscore
vspm
totalgold
earnedgold
earned gpm
earnedgoldshare
goldspent

## Step 1: Introduction

In [5]:
# TODO

## Step 2: Data Cleaning and Exploratory Data Analysis

In [6]:
# TODO

## Step 3: Assessment of Missingness

In [7]:
# TODO

## Step 4: Hypothesis Testing

In [8]:
blue_win_rate = (df["side"] == "Blue").mean()

out = []
for _ in range(10000):
    sims = np.random.multinomial(df.shape[0], [0.5, 0.5])
    proportion = sims[0] / df.shape[0]   
    out.append(proportion)

out = np.array(out)

pval = np.mean(out >= blue_win_rate)
pval

np.float64(0.5032)

In [9]:
if pval > 0.05:
    print("we do not reject the null hypothesis with a p-value of " + str(pval))
else:
    print("we reject the null with a p-value of " + str(pval))

we do not reject the null hypothesis with a p-value of 0.5032


## Step 5: Framing a Prediction Problem

In [10]:
# TODO

## Step 6: Baseline Model

In [11]:
df["kill_participation"] = (df["kills"] + df["assists"]) / df["teamkills"]
df["damage_efficiency"] = df["damagetochampions"] / (df["deaths"] + 1)

df["carry_impact"] = (
    0.35 * df["kill_participation"] +
    0.30 * df["damageshare"] +
    0.20 * df["earnedgoldshare"] +
    0.10 * df["damage_efficiency"] +
    0.05 * df["visionscore"]
)

team_carry = df.groupby(["gameid", "side"])["carry_impact"].sum().unstack()
team_carry["carry_gap"] = team_carry["Blue"] - team_carry["Red"]

df["playername"].value_counts().head(10)

df.groupby("position")["carry_impact"].mean().sort_values(ascending=False)

position
bot     915.999917
mid     798.676266
top     644.004290
jng     488.935332
sup     198.747027
team           NaN
Name: carry_impact, dtype: float64

The above is a hard-coded determination of the feature weights to make up the carry score. The final model will use these same feature but decides the weights of each of these features to predict win outcome

## Step 7: Final Model

In [18]:
import numpy as np

df["kill_participation"] = (df["kills"] + df["assists"]) / df["teamkills"]
df["damage_efficiency"] = df["damagetochampions"] / (df["deaths"] + 1)

df["carry_impact"] = (
    0.35 * df["kill_participation"] +
    0.30 * df["damageshare"] +
    0.20 * df["earnedgoldshare"] +
    0.10 * df["damage_efficiency"] +
    0.05 * df["visionscore"]
)

features = [
    "kill_participation",
    "damageshare",
    "earnedgoldshare",
    "damage_efficiency",
    "visionscore",
    "carry_impact"
]

# keep only needed columns
model_df = df[features + ["result"]].copy()

# replace inf with NaN
model_df = model_df.replace([np.inf, -np.inf], np.nan)

# drop missing rows
model_df = model_df.dropna()

X = model_df[features]
y = model_df["result"]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
log_preds = log_model.predict(X_test)

tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)
tree_preds = tree_model.predict(X_test)

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

print("Logistic Regression:", accuracy_score(y_test, log_preds))
print("Decision Tree:", accuracy_score(y_test, tree_preds))
print("Random Forest:", accuracy_score(y_test, rf_preds))

Logistic Regression: 0.8065737051792828
Decision Tree: 0.7681772908366534
Random Forest: 0.8313745019920319


: 

Random Forest Model 

## Step 8: Fairness Analysis

In [13]:
# TODO

This model can be used to predict the outcome of future games by inputting player or team statistics into the trained classifier. The model analyzes patterns learned from historical match data to estimate whether a team is likely to win or lose based on metrics such as damage share, gold share, and kill participation.